## Import Libraries

In [1]:
import os
#import cv2
import math
import time
import tqdm
import yaml
#import tarfile
#import numbers
#import threading
#import queue as Queue
import numpy as np
import pandas as pd
from PIL import Image
from random import random
import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.datasets import ImageFolder
from torchvision.datasets.utils import download_url
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import random_split, DataLoader, Dataset, TensorDataset
from torchsummary import summary

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
random_seed = 42
torch.manual_seed(random_seed);

In [4]:
torch.set_printoptions(edgeitems=5)

In [5]:
def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def to_device(data, device):
    """Move tensor(s) to chosen device"""
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True) #, dtype=torch.float

class DeviceDataLoader():
    """Wrap a dataloader to move data to a device"""
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device
        
    def __iter__(self):
        """Yield a batch of data after moving it to device"""
        for b in self.dl: 
            yield to_device(b, self.device)

    def __len__(self):
        """Number of batches"""
        return len(self.dl)

In [6]:
device = get_default_device()

In [7]:
device

device(type='cuda')

## Dataset Processing

In [8]:
# yml_file = "data.yml" #mydata.yml #data.yml
# data_dir = "all_data" #all_data2 #all_data

In [9]:
yml_file = "traindata2.yml" #mydata.yml #data.yml
data_dir = "MultiPIE-3" #all_data2 #all_data

root_dir = '/home/barc/Desktop/subir/Projects/TP-GAN'

In [10]:
images_list = os.path.join(root_dir, yml_file)
images_dir = os.path.join(root_dir, data_dir)

In [11]:
save_dir = "generated_data_pair_func"
images_save_dir = os.path.join(root_dir, save_dir)

if not os.path.exists(images_save_dir):
    os.makedirs(images_save_dir)

In [12]:
test_yml_file = "testdata2.yml" #mydata.yml #data.yml
data_dir2 = "MultiPIE-3" #all_data2 #all_data

test_images_list = os.path.join(root_dir, test_yml_file)
test_images_dir = os.path.join(root_dir, data_dir2)

In [13]:
def createDataset(images_list, images_dir, p_test=0.1):
    with open(images_list, 'r') as rf:
        images_list_all = yaml.safe_load(rf.read())
    
    images_list_test = list()
    images_list_train = list()

    if p_test == 1:
        for k in images_list_all.keys():
            if images_list_all[k]['img'] != images_list_all[k]['imgGT']:
                images_list_test.append(k)
    else:
        nb_total = len(images_list_all)
        print(nb_total)
        counter = 0
        
        for k in images_list_all.keys():
            if counter < nb_total*p_test:
                images_list_test.append(k)
                counter += 1
            else:
                images_list_train.append(k)
    
    
    return CustomDataset(images_list_train, images_list_all, images_dir)
#, CustomDataset(images_list_test, images_list_all, images_dir)

In [14]:
class CustomDataset(Dataset):
    def __init__(self, images_list_selected, images_list_all, images_dir):
        super(CustomDataset, self).__init__()
        self.images_list_selected = images_list_selected
        self.images_list = images_list_all
        self.images_dir = images_dir
        self.keys = list(self.images_list_selected)
        
    def __len__(self):
        return len(self.images_list_selected)
    
    def __getitem__(self, idx):
        # Return a dict with :
        #  - profile and frontal (ground truth) images with size 128x128, 64x64 and 32x32
        
        stats = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

        image_info = self.images_list[self.keys[idx]]
        image   = Image.open(os.path.join(self.images_dir, image_info['img']))
        imageGT = Image.open(os.path.join(self.images_dir, image_info['imgGT']))
        
        batch = dict()
        
        batch['id']       = int(image_info['id'])
        batch['img128']   = image
        batch['img64']    = transforms.functional.resize(image, (64, 64))
        batch['img32']    = transforms.functional.resize(image, (32, 32))
        batch['img128_GT'] = imageGT
        batch['img64_GT']  = transforms.functional.resize(imageGT, (64, 64))
        batch['img32_GT']  = transforms.functional.resize(imageGT, (32, 32))
        
        transform = transforms.Compose([
            #transforms.Resize(128),
            transforms.ToTensor(),
            transforms.Normalize(*stats)
        ])
        
        for k in batch.keys():
            if (k == 'id'):
                continue
            batch[k] = transform(batch[k])
            #print('{} : {}'.format(k, batch[k].shape))
        
        return batch

In [15]:
dataset = createDataset(images_list, images_dir)

18821


In [16]:
# plt.rcParams["figure.figsize"] = (5, 5)

In [17]:
# for d in dataset:     
#     img1 = torch.permute(d['img128_GT'], (1, 2, 0))
#     img2 = torch.permute(d['img128'], (1, 2, 0))
    
#     #print(torch.min(img1))
#     #print(torch.max(img1))
    
#     img1 = (img1*127)+127.5
#     img1 = img1.type(torch.int64)
#     img2 = (img2*127)+127.5
#     img2 = img2.type(torch.int64)
    
    
#     #print(torch.min(img1))
#     #print(torch.max(img1))

#     #Normalized image
#     plt.subplot(1, 2, 1)
#     plt.imshow(img1)
#     plt.axis('off')
#     plt.title('Id: ' + str(d['id']))

#     plt.subplot(1, 2, 2)
#     plt.imshow(img2)
#     plt.axis('off')

#     plt.show()

In [18]:
# ids = []

# for d in dataset:
#     id = d['id']
#     if id not in ids:
#         ids.append(id)
        
#         img1 = torch.permute(d['img128'], (1, 2, 0))
#         img2 = torch.permute(d['img128_GT'], (1, 2, 0))

#         img1 = (img1*127)+127.5
#         img1 = img1.type(torch.int64)
        
#         img2 = (img2*127)+127.5
#         img2 = img2.type(torch.int64)

#         #Normalized image
#         plt.subplot(1, 2, 1)
#         plt.imshow(img1)
#         plt.axis('off')
#         plt.title('Id: ' + str(d['id']))

#         plt.subplot(1, 2, 2)
#         plt.imshow(img2)
#         plt.axis('off')

#         plt.show()

# print(len(ids))

In [19]:
dataset

In [20]:
len(dataset)

16938

In [21]:
bs = 50

In [22]:
train_dl = DataLoader(dataset, batch_size=bs, shuffle=True, num_workers=4, pin_memory=True) #34
#train_dl = DeviceDataLoader(train_ds, device)
len(train_dl)

339

In [23]:
test_dataset = createDataset(test_images_list, test_images_dir)

557


In [24]:
# ids = []

# for d in test_dataset:
#     id = d['id']
#     if id not in ids:
#         ids.append(id)
        
#         img1 = torch.permute(d['img128'], (1, 2, 0))
#         img2 = torch.permute(d['img128_GT'], (1, 2, 0))

#         img1 = (img1*127)+127.5
#         img1 = img1.type(torch.int64)
        
#         img2 = (img2*127)+127.5
#         img2 = img2.type(torch.int64)

#         #Normalized image
#         plt.subplot(1, 2, 1)
#         plt.imshow(img1)
#         plt.axis('off')
#         plt.title('Id: ' + str(d['id']))

#         plt.subplot(1, 2, 2)
#         plt.imshow(img2)
#         plt.axis('off')

#         plt.show()

# print(len(ids))

In [25]:
test_dl = DataLoader(test_dataset, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True) #34
#test_dl = DeviceDataLoader(test_dl, device)
len(test_dl)

11

In [26]:
for inputs in train_dl:
    print(inputs['img128'].shape, inputs['img64'].shape, inputs['img32'].shape)
    break

torch.Size([50, 3, 128, 128]) torch.Size([50, 3, 64, 64]) torch.Size([50, 3, 32, 32])


In [27]:
# for b in dataset:
#     print(b['id'])
#     print(b['img128'].shape)
#     print(b['img64'].shape)
#     print(b['img32'].shape)
#     print(b['img128_GT'].shape)
#     print(b['img64_GT'].shape)
#     print(b['img32_GT'].shape)
#     print("--------------")
#     break

## Classification Network

In [28]:
def bn_relu_conv(ni, nf, ks):
    return nn.Sequential(nn.BatchNorm2d(ni), 
                       nn.ReLU(inplace=True),
                       conv_2d_s1(ni, nf, ks))

def conv_2d_s1(ni, nf, ks, stride=1):
    return nn.Conv2d(in_channels=ni, out_channels=nf, kernel_size=ks, stride=stride, padding=ks//2, bias=False)

In [29]:
class ResBlock1(nn.Module):
    def __init__(self, ni, nf, stride=1):
        super().__init__()
        if ni > 100:
            temp = ni * 2
        else:
            temp = ni
        self.bn = nn.BatchNorm2d(temp)
        self.conv1 = conv_2d_s1(temp, ni, 1, stride)
        self.conv2 = bn_relu_conv(ni, ni, ks=3)
        self.conv3 = bn_relu_conv(ni, nf, ks=1)
        self.shortcut = lambda x: x
        if ni != nf:
            self.shortcut = conv_2d_s1(temp, nf, 1, stride)

    def forward(self, x):
        #print("Inside Res Block1")
        #print(x.shape)
        x = F.relu(self.bn(x), inplace=True)
        #print(x.shape)
        r1 = self.shortcut(x)
        #print(r1.shape)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x) * 0.2
        #print(x.shape)
        return x.add_(r1)

In [30]:
class ResBlock2(nn.Module):
    def __init__(self, ni, nf, stride=1):
        super().__init__()
        self.bn = nn.BatchNorm2d(ni)
        self.conv1 = conv_2d_s1(ni, nf, 1, stride)
        self.conv2 = bn_relu_conv(nf, nf, ks=3)
        self.conv3 = bn_relu_conv(nf, ni, ks=1)
        self.shortcut = lambda x: x
#        if ni != nf:
#            self.shortcut = conv_2d(ni, nf, 1, 1)

    def forward(self, x):
        #print("Inside Res Block2")
        #print(x.shape)
        x = F.relu(self.bn(x), inplace=True)
        #print(x.shape)
        r = self.shortcut(x)
        #print(r.shape)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x) * 0.2
        return x.add_(r)

In [31]:
def make_group(N, ni, nf, stride):
    start = ResBlock1(ni, nf, stride)
    rest = [ResBlock2(nf, ni) for j in range(1, N)]
    return [start] + rest

In [32]:
class Flatten(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return x.view(x.size(0), -1)

In [33]:
class MyResNet(nn.Module):
    def __init__(self, n_groups, N, k=1, n_start=64):
        super().__init__()
        #Increase channels
        self.layers = [conv_2d_s1(3, 64, ks=7, stride=2)]
        self.layers += [nn.MaxPool2d(kernel_size=3, stride=2, padding=1)]
        n_channels = [n_start]

        #Add groups
        for i in range(n_groups):
            n_channels.append(n_start*(2**i)*k)
            stride = 2 if i>0 else 1
            self.layers += make_group(N[i], n_channels[i], n_channels[i]*4, stride)

        #Pool, Flatten, and add linear layer for classification  
        self.layers += [nn.BatchNorm2d(n_channels[n_groups]*2),
                        nn.ReLU(inplace=True),
                        nn.AdaptiveAvgPool2d(1),
                        #nn.AvgPool2d(kernel_size=2, stride=2),
                        Flatten(),
                        nn.Linear(n_channels[n_groups]*2, 512)
                       ]
        #self.fc = nn.Linear(512, n_classes)
        self.features = nn.Sequential(*self.layers)
        
    def forward(self, x):
        embed = self.features(x)
        #print(embed.shape)
        return embed #self.fc(embed)

In [34]:
#Number of blocks at various groups
N_50 = [3, 4, 6, 3]

def ResNet50():
    return MyResNet(4, N_50, k=2)

In [35]:
# def test():
#     net = ResNet50()
#     x = torch.randn(2, 3, 128, 128)
#     y = net(x)
#     print(y.shape)
#     to_device(net, device)
#     summary(net, input_size = (3, 128, 128), batch_size = -1)
#     return net

In [36]:
#test()

In [37]:
class CosFace(nn.Module):
    def __init__(self, in_features=2048, out_features=5749, s=64.0, m=0.35):
        super(CosFace, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.kernel = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.normal_(self.kernel, std=0.01)

    def forward(self, logits, labels):
        logits = F.normalize(logits, p=2.0, dim=1) #l2_norm(logits, axis=1)
        kernel_norm = F.normalize(self.kernel, p=2.0, dim=0) #l2_norm(self.kernel, axis=0)
        cos_theta = torch.mm(logits, kernel_norm)
        cos_theta = cos_theta.clamp(-1, 1)  # for numerical stability
        index = torch.where(labels != -1)[0]
        m_hot = torch.zeros(index.size()[0], cos_theta.size()[1]).to(device)
        m_hot.scatter_(1, labels[index, None], self.m).to(device)
        cos_theta[index] -= m_hot
        ret = cos_theta * self.s
        return ret

In [38]:
# resnet_pth = "/home/barc/Desktop/subir/ElasticFace/Checkpoints/ResNet50_CosFacev2/512-d/ResNet50_CosFace_14_Mar_15.pt"
# checkpoint = torch.load(resnet_pth)
# resNet.load_state_dict(checkpoint['model_state_dict'])
# epoch = checkpoint['epoch']
# train_acc = checkpoint['train_acc']
# val_acc = checkpoint['val_acc']
# print(epoch, train_acc, val_acc)

In [39]:
# for param in resNet.parameters():
#     param.requires_grad = False

In [40]:
# to_device(resNet, device)
# summary(resNet, input_size = (3, 112, 112), batch_size = -1, device="cuda")

## Network Architecture

In [41]:
def relu():
    return nn.ReLU()

def lrelu(f=0.2):
    return nn.LeakyReLU(f)

def tanh():
    return nn.Tanh()

def batch_norm(ni):
    return nn.BatchNorm2d(ni)

def conv_2d(ni, nf, ks, stride=2):
    return nn.Conv2d(in_channels=ni, out_channels=nf, kernel_size=ks, stride=stride, padding=ks//2, bias=False)

def deconv_2d(ni, nf, ks, stride=2, padding=1, output_padding=1):
    return nn.ConvTranspose2d(in_channels=ni, out_channels=nf, 
                               kernel_size=ks, stride=stride, 
                               padding=padding, output_padding=output_padding)
    
def fc_nn(input_size, output_size):
    return nn.Sequential(nn.Flatten(), 
                          nn.Linear(input_size, output_size)
                         )

In [42]:
class ResBlock(nn.Module):
    def __init__(self, ni, ks=3, stride=1):
        super().__init__()
        self.conv = conv_2d(ni, ni, ks, stride)
        self.bn = batch_norm(ni)
        self.lrelu = lrelu()
        self.shortcut = lambda x: x

    def forward(self, x):
        r = self.shortcut(x)
        x = self.conv(x)
        x = self.bn(x)
        x = self.lrelu(x)
        x = self.conv(x)
        x = self.bn(x)
        x = self.lrelu(x.add_(r))
        return x

In [43]:
def show_shapes(*feats):
    for f in feats:
        print(f.shape)

### Generator

In [44]:
class Generator(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
#         self.path_leye = GeneratorLocal()
#         self.path_reye = GeneratorLocal()
#         self.path_nose = GeneratorLocal()
#         self.path_mouth = GeneratorLocal()
#         self.fuser = LocalFuser()
        
        self.globalpath = GeneratorGlobal()
        #self.feature_predict = FeaturePredict(num_classes)
        
        
    def forward(self, img128, img64, img32):
                                            #, leye, reye, nose, mouth
                                            #, noise):
        ## --- Move to device
        if(img128.is_cuda == False):
            img128 = img128.to(device)
        if(img64.is_cuda == False):
            img64 = img64.to(device)
        if(img32.is_cuda == False):
            img32 = img32.to(device)
        
#         #Local Path
#         fake_leye, fake_leye_features = self.path_leye(leye)
#         fake_reye, fake_reye_features = self.path_reye(reye)
#         fake_nose, fake_nose_features = self.path_nose(nose)
#         fake_mouth, fake_mouth_features = self.path_mouth(mouth)
        
#         #Merge Local Path
#         local_features = self.fuser(fake_leye_features, fake_reye_features, fake_nose_features, 
#                                     fake_mouth_features)
#         local_fake = self.fuser(fake_leye, fake_reye, fake_nose, fake_mouth)
#         local_GT = self.fuser(leye, reye, nose, mouth)
        
        #Global Path
#         fake_img128, fake_img64, fake_img32, fc2 = self.globalpath(img128, img64, img32, 
#                                                                    local_fake, local_features) #, noise)
        fake_img128, fake_img64, fake_img32, fc2 = self.globalpath(img128, img64, img32) #, noise)
#         encoder_predict = self.feature_predict(fc2)
        
#         return fake_img128, fake_img64, fake_img32, encoder_predict, \
#                 local_fake, fake_leye, fake_reye, fake_nose, fake_mouth, local_GT

        return fake_img128, fake_img64, fake_img32 #, encoder_predict

In [45]:
# img128 = torch.randn(batch_size, 3, 128, 128).to(device)
# img64 = torch.randn(batch_size, 3, 64, 64).to(device)
# img32 = torch.randn(batch_size, 3, 32, 32).to(device)

# # leye = torch.randn(batch_size, 3, 128, 128)
# # reye = torch.randn(batch_size, 3, 128, 128)
# # nose = torch.randn(batch_size, 3, 128, 128)
# # mouth = torch.randn(batch_size, 3, 128, 128)

# #noise = torch.randn(batch_size, 256)

In [46]:
# model = Generator(10)
# to_device(model, device)

# out = model(img128, img64, img32) #, leye, reye, nose, mouth)#, noise)

In [47]:
# show_shapes(*out)

### Generator: Global

In [48]:
class GeneratorGlobal(nn.Module):
    def __init__(self):
        super().__init__()
        
        dim = [3, 64, 128, 256, 512]
        dec = [64, 32, 16, 8]
        
        
        #Encoder
        #---------------
        
        self.conv0 = nn.Sequential(
                    conv_2d(dim[0], dim[1], ks=7, stride=1),
                    lrelu(),
                    ResBlock(dim[1], ks=7))
        
        self.conv1 = nn.Sequential(
                    conv_2d(dim[1], dim[1], ks=5, stride=2),
                    batch_norm(dim[1]),
                    lrelu(),
                    ResBlock(dim[1], ks=5))
        
        self.conv2 = nn.Sequential(
                    conv_2d(dim[1], dim[2], ks=3, stride=2),
                    batch_norm(dim[2]),
                    lrelu(),
                    ResBlock(dim[2], ks=3))
        
        self.conv3 = nn.Sequential(
                    conv_2d(dim[2], dim[3], ks=3, stride=2),
                    batch_norm(dim[3]),
                    lrelu(),
                    ResBlock(dim[3], ks=3))
        
        self.conv4 = nn.Sequential(
                    conv_2d(dim[3], dim[4], ks=3, stride=2),
                    batch_norm(dim[4]),
                    lrelu(),
                    ResBlock(dim[4], ks=3),
                    ResBlock(dim[4], ks=3),
                    ResBlock(dim[4], ks=3),
                    ResBlock(dim[4], ks=3))
        
        self.fc1 = nn.Sequential(
                    fc_nn(dim[1]*dim[4], dim[4]))

        
        
        #Decoder
        #---------------
        
        #Layer-feat8 [bs, 64, 8, 8]
        self.feat8_ = nn.Sequential(
                    fc_nn(dim[4], dim[1]*8*8))
        self.feat8 = nn.Sequential(
                    relu())
        
        #Layer-feat32 [bs, 32, 32, 32]
        self.feat32 = nn.Sequential(
                    deconv_2d(dec[0], dec[1], 3, 4, 0, 1),
                    relu())
        
        #Layer-feat64 [bs, 16, 64, 64]
        self.feat64 = nn.Sequential(
                    deconv_2d(dec[1], dec[2], 3, 2, 1, 1),
                    relu())
        
        #Layer-feat128 [bs, 8, 128, 128]
        self.feat128 = nn.Sequential(
                    deconv_2d(dec[2], dec[3], 3, 2, 1, 1),
                    relu())
    
        #Layer - deconv0 [bs, 512, 16, 16]
        self.deconv0_16 = nn.Sequential(
                    ResBlock(ni=576),
                    ResBlock(ni=576),
                    ResBlock(ni=576),
                    deconv_2d(576, dim[4], 3, 2, 1, 1),
                    batch_norm(dim[4]),
                    relu())
        
        #Layer - deconv1 [bs, 256, 32, 32]
        self.decode_16 = nn.Sequential(
                    ResBlock(ni=256))
        
        self.deconv1_32 = nn.Sequential(
                    ResBlock(ni=768),
                    ResBlock(ni=768),
                    deconv_2d(768, dim[3], 3, 2, 1, 1),
                    batch_norm(dim[3]),
                    relu())
        
        #Layer - deconv2 [bs, 128, 64, 64]
        self.decode_32 = nn.Sequential(
                    ResBlock(ni=163))
        
        self.reconstruct_32 = nn.Sequential(
                    ResBlock(ni=419),
                    ResBlock(ni=419))
        
        self.deconv2_64 = nn.Sequential(
                    deconv_2d(419, dim[2], 3, 2, 1, 1),
                    batch_norm(dim[2]),
                    relu())
        
        self.img32 = nn.Sequential(
                    conv_2d(ni=419, nf=dim[0], ks=3, stride=1),
                    tanh())
        
        #Layer - deconv3 [bs, 64, 128, 128]
        self.decode_64 = nn.Sequential(
                    ResBlock(ni=83, ks=5))
        
        self.reconstruct_64 = nn.Sequential(
                    ResBlock(ni=214),
                    ResBlock(ni=214))
        
        self.deconv3_128 = nn.Sequential(
                    deconv_2d(214, dim[1], 3, 2, 1, 1),
                    batch_norm(dim[1]),
                    relu())
        
        self.img64 = nn.Sequential(
                    conv_2d(ni=214, nf=dim[0], ks=3, stride=1),
                    tanh())
        
        #Layer - conv5 [bs, 64, 128, 128]
        self.decode_128 = nn.Sequential(
                    ResBlock(ni=75, ks=7))
        
        self.reconstruct_128 = nn.Sequential(
                    ResBlock(ni=142, ks=5)) #ni=209
        
        self.conv5 = nn.Sequential(
                    conv_2d(142, dec[0], ks=5, stride=1), #209
                    batch_norm(dec[0]),
                    lrelu(),
                    ResBlock(ni=dec[0]))

        #Layer - conv6 [bs, 32, 128, 128]
        self.conv6 = nn.Sequential(
                    conv_2d(dec[0], dec[1], ks=3, stride=1),
                    batch_norm(dec[1]),
                    lrelu())
        
        #Layer - conv7 [bs, 3, 128, 128]
        self.img128 = nn.Sequential(
                    conv_2d(ni=dec[1], nf=dim[0], ks=3, stride=1),
                    tanh())

    
    def forward(self, I_P_128, I_P_64, I_P_32): #, local_predict, local_feature): #, noise):
        ## --- Move to device
        if(I_P_128.is_cuda == False):
            I_P_128 = I_P_128.to(device)
        if(I_P_64.is_cuda == False):
            I_P_64 = I_P_64.to(device)
        if(I_P_32.is_cuda == False):
            I_P_32 = I_P_32.to(device)
            
        bs = I_P_64.shape[0]
        noise = torch.randn(bs, 256).to(device)
        #print(noise.is_cuda)
        
        conv0 = self.conv0(I_P_128)
        conv1 = self.conv1(conv0)
        conv2 = self.conv2(conv1)
        conv3 = self.conv3(conv2)
        conv4 = self.conv4(conv3)
        #print(conv4.shape)
        fc1 = self.fc1(conv4)
        fc2 = torch.maximum(fc1[:, 0:256], fc1[:, 256:])
        #print(fc2.is_cuda)
        
        feat8_ = self.feat8_(torch.cat((fc2, noise), 1)).view(fc2.size()[0], 64, 8, 8)
                                                             #Output: [bs, 64, 8, 8]
        
        feat8 = self.feat8(feat8_) #Output: [bs, 64, 8, 8]
        
        feat32 = self.feat32(feat8) #Output: [bs, 32, 32, 32]
        
        feat64 = self.feat64(feat32) #Output: [bs, 16, 64, 64]
        
        feat128 = self.feat128(feat64) #Output: [bs, 8, 128, 128]
        
        deconv0_16 = self.deconv0_16(torch.cat((feat8, conv4), 1)) #Output: [bs, 512, 16, 16]
        
        decode_16 = self.decode_16(conv3)
        deconv1_32 = self.deconv1_32(torch.cat((deconv0_16, decode_16), 1)) #Output: [bs, 256, 32, 32]
        
        decode_32 = self.decode_32(torch.cat((conv2, feat32, I_P_32), 1))
        reconstruct_32 = self.reconstruct_32(torch.cat((deconv1_32, decode_32), 1))
        deconv2_64 = self.deconv2_64(reconstruct_32) #Output: [bs, 128, 64, 64]
        img32 = self.img32(reconstruct_32) #Output: [bs, 3, 32, 32]
        
        decode_64 = self.decode_64(torch.cat((conv1, feat64, I_P_64), 1))
        reconstruct_64 = self.reconstruct_64(torch.cat((deconv2_64, decode_64, \
                                                       F.interpolate(img32, (64,64), \
                                                                     mode='bilinear', align_corners=False)), 1))
        deconv3_128 = self.deconv3_128(reconstruct_64) #Output: [bs, 64, 128, 128]
        img64 = self.img64(reconstruct_64) #Output: [bs, 3, 64, 64]
        
        decode_128 = self.decode_128(torch.cat((conv0, feat128, I_P_128), 1))
        #Concatenated eyel, eyer, nose, mouth, c_eyel, c_eyer, c_nose, c_mouth
        reconstruct_128 = self.reconstruct_128(torch.cat((deconv3_128, decode_128, \
                                                       F.interpolate(img64, (128,128), \
                                                                     mode='bilinear', align_corners=False)), 1))
                                                                            #, local_feature, local_predict), 1))
        
        conv5 = self.conv5(reconstruct_128) #Output: [bs, 64, 128, 128]
        
        conv6 = self.conv6(conv5) #Output: [bs, 32, 128, 128]
        
        img128 = self.img128(conv6) #Output: [bs, 3, 128, 128]
        
        
        return img128, img64, img32, fc2

In [49]:
# #input1 = torch.randn(batch_size, 3, 128, 128).to(device) #local_fake
# #input2 = torch.randn(batch_size, 64, 128, 128).to(device) #local_feature
# #noi = torch.randn(batch_size, 256)

# I_P_32 = torch.randn(batch_size, 3, 32, 32).to(device)
# I_P_64 = torch.randn(batch_size, 3, 64, 64).to(device)
# I_P_128 = torch.randn(batch_size, 3, 128, 128).to(device)


# model = GeneratorGlobal()
# to_device(model, device)

# feats = model(I_P_128, I_P_64, I_P_32) #, input1, input2)#, noi)
# #feats.shape

In [50]:
#feats.shape
#feats

In [51]:
# input1 = torch.randn(batch_size, 3, 128, 128).to(device) #local_fake
# input2 = torch.randn(batch_size, 64, 128, 128) #local_feature

# input1.is_cuda, input2.is_cuda

In [52]:
# help(summary)

In [53]:
# # summary(model, input_size = [(3, 128, 128), (3, 64, 64), (3, 32, 32), 
# #                              (3, 128, 128), (64, 128, 128)], batch_size = -1, device='cuda') #'cpu'
# summary(model, input_size = [(3, 128, 128), (3, 64, 64), (3, 32, 32)], 
#         batch_size = -1, device='cuda') #'cpu'

In [54]:
# show_shapes(*feats)

### Discriminator

In [55]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        dim = [3, 64, 128, 256, 512]
        
        self.conv0 = nn.Sequential(
                    conv_2d(dim[0], dim[1], ks=3, stride=2),
                    batch_norm(dim[1]),
                    lrelu())
        
        self.conv1 = nn.Sequential(
                    conv_2d(dim[1], dim[2], ks=3, stride=2),
                    batch_norm(dim[2]),
                    lrelu())
        
        self.conv2 = nn.Sequential(
                    conv_2d(dim[2], dim[3], ks=3, stride=2),
                    batch_norm(dim[3]),
                    lrelu())
        
        self.conv3 = nn.Sequential(
                    conv_2d(dim[3], dim[4], ks=3, stride=2),
                    batch_norm(dim[4]),
                    lrelu(),
                    ResBlock(ni=dim[4]))
        
        self.conv4 = nn.Sequential(
                    conv_2d(dim[4], dim[4], ks=3, stride=2),
                    batch_norm(dim[4]),
                    lrelu(),
                    ResBlock(ni=dim[4]))
        
        self.conv5 = nn.Sequential(
#                     conv_2d(dim[4], 1, ks=3, stride=1)
                    nn.Conv2d(in_channels=dim[4], out_channels=1, kernel_size=4, stride=1, padding=0, bias=False),
                    nn.Flatten(),
                    nn.Sigmoid()
                    )
        
        
    def forward(self, inp_image):
        ## --- Move to device
        if(inp_image.is_cuda == False):
            inp_image = inp_image.to(device)
        
        x = self.conv0(inp_image)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
#         x = x.view(x.size()[0], -1)
        
        return x

In [56]:
# input2 = torch.randn(10, 3, 128, 128)
# model = Discriminator()
# output2 = model(input2)

# output2.shape

## Loss Functions

In [57]:
def cantor_pair(x, y):
    loss = 0.5*(x*x + x + 2*x*y + y*y + 3*y)
    return loss

In [58]:
class G_Loss(nn.Module):
    def __init__(self):
        super().__init__()
        alpha = 0.001
        self.L1loss = nn.L1Loss()
        self.CrossEntropy = nn.CrossEntropyLoss()
        
        self.resNet = ResNet50()
        to_device(self.resNet, device)
        resnet_pth = "/home/barc/Desktop/subir/ElasticFace/Checkpoints/ResNet50_CosFacev2/512-d/ResNet50_CosFace_14_Mar_15.pt"
        checkpoint = torch.load(resnet_pth)
        self.resNet.load_state_dict(checkpoint['model_state_dict'])
        for param in self.resNet.parameters():
            param.requires_grad = False

    def pixel_wise_loss(self, img128_fake, img64_fake, img32_fake, inputs):
        
        ## --- Move to device
        if(inputs['img128_GT'].is_cuda == False):
            inputs['img128_GT'] = inputs['img128_GT'].to(device)
        if(inputs['img64_GT'].is_cuda == False):
            inputs['img64_GT'] = inputs['img64_GT'].to(device)
        if(inputs['img32_GT'].is_cuda == False):
            inputs['img32_GT'] = inputs['img32_GT'].to(device)
            
        l128 = self.L1loss(img128_fake, inputs['img128_GT'])
        l64  = self.L1loss(img64_fake, inputs['img64_GT'])
        l32  = self.L1loss(img32_fake, inputs['img32_GT'])
        global_loss = l128 +l64 +l32
        
        return global_loss
    
    
    def symmetry_loss(self, img128_fake, img64_fake, img32_fake):
        img128_fake_mirror = img128_fake.index_select(3, torch.arange(img128_fake.size()
                                                                      [3]-1, -1, -1).long().to(device))
        img128_fake_mirror.detach_()
        img64_fake_mirror = img64_fake.index_select(3, torch.arange(img64_fake.size()
                                                                    [3]-1, -1, -1).long().to(device))
        img64_fake_mirror.detach_()
        img32_fake_mirror = img32_fake.index_select(3, torch.arange(img32_fake.size()
                                                                    [3]-1, -1, -1).long().to(device))
        img32_fake_mirror.detach_()
        
        symloss128 = self.L1loss(img128_fake, img128_fake_mirror)
        symloss64 = self.L1loss(img64_fake, img64_fake_mirror)
        symloss32 = self.L1loss(img32_fake, img32_fake_mirror)
        
        return symloss128 + symloss64 + symloss32
        
    
    def adversarial_loss(self, D, img128_fake):
        return -torch.mean(D(img128_fake))
    
    
    def identity_preserving_loss(self, img128_fake, inputs):
        ## --- Move to device
        if(inputs['img128_GT'].is_cuda == False):
            inputs['img128_GT'] = inputs['img128_GT'].to(device)
            
        fake_embed = self.resNet(img128_fake)
        real_embed = self.resNet(inputs['img128_GT'])
        
        return self.L1loss(fake_embed, real_embed)
        
    
    def total_variation_loss(self, img128_fake):
        
        return torch.mean(
            torch.abs(img128_fake[:,:,:-1,:] - img128_fake[:,:,1:,:])) + torch.mean(
            torch.abs(img128_fake[:,:,:,:-1] - img128_fake[:,:,:,1:]))
    
           
    def forward(self, G, D, img128_fake, img64_fake, img32_fake, inputs): #encoder_predict, 
                
        L_pixel  = self.pixel_wise_loss(img128_fake, img64_fake, img32_fake, inputs)
        L_sym = self.symmetry_loss(img128_fake, img64_fake, img32_fake)
        L_adv = self.adversarial_loss(D, img128_fake)
        L_ip  = self.identity_preserving_loss(img128_fake, inputs)
        L_tv  = self.total_variation_loss(img128_fake)
        
#        L_syn = L_pixel + 0.3*L_sym + 0.001*L_adv + 0.003*L_ip + 0.0001*L_tv
        L_sym = 0.8 * L_sym
        L_adv = 0.002 * L_adv
        L_ip = 0.008 * L_ip
        L_tv = 0.0001 * L_tv
        L_syn = cantor_pair(L_pixel, cantor_pair(L_sym, cantor_pair(L_adv, cantor_pair(L_ip, L_tv))))
        
        loss_gen = L_syn
        
        return loss_gen

In [59]:
# test1 = G_Loss()
# to_device(test1, device)

# out1 = test1(G, D, img128_fake, img64_fake, img32_fake, inputs) #encoder_predict, 
#                 #local_fake, left_eye_fake, right_eye_fake, nose_fake, mouth_fake, local_GT, inputs)

In [60]:
# print(out1.shape)
# print(out1)

In [61]:
class D_Loss(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, D, img128_fake, inputs):
        ## --- Move to device
        if(inputs['img128_GT'].is_cuda == False):
            inputs['img128_GT'] = inputs['img128_GT'].to(device)
            
        adv_D_loss = torch.mean(D(img128_fake.detach())) - torch.mean(D(inputs['img128_GT']))
        
        alpha = torch.rand(inputs['img128_GT'].shape[0], 1, 1, 1).expand_as(inputs['img128_GT']).to(device)
        
        interpolated_x = Variable(alpha * img128_fake.detach().data.to(device) + 
                                  (1.0 - alpha) * inputs['img128_GT'].data.to(device), requires_grad = True)
        
        out = D(interpolated_x)
        
        dxdD = torch.autograd.grad(outputs = out, inputs = interpolated_x, 
                                   grad_outputs = torch.ones(out.size()).to(device), 
                                   retain_graph = True, create_graph = True, 
                                   only_inputs = True)[0].view(out.shape[0],-1)
        
        gp_loss = torch.mean((torch.norm(dxdD, p = 2) - 1)**2)
        
        return adv_D_loss + 10*gp_loss

In [62]:
# test2 = D_Loss()
# to_device(test2, device)

# out = test2(D, img128_fake, inputs)
# print(out.shape)
# print(out)

### Save samples

In [63]:
def save_samples(index, G, data_dl, show=False):
    for each_batch in data_dl:
        #img1 = torch.permute(each_batch['img128'][0], (1, 2, 0))
        #img1 = (img1*255)
        #img1 = img1.type(torch.int64)
        
        with torch.no_grad():
            #Generate predictions
            img128_fake, img64_fake, img32_fake = G(each_batch['img128'], each_batch['img64'], each_batch['img32'])
            #img2 = torch.permute(img128_fake[0], (1, 2, 0))
            #img2 = (img2*127)+127.5
            
            #img2 = img2.detach().cpu()
            #img2 = img2.type(torch.int64)
            #print(img2)
            
            #new_pair = torch.stack((each_batch['img128'][0].detach().cpu(), img128_fake[0].detach().cpu()))
            new_pair = torch.stack((each_batch['img128'][0].detach().cpu(), 
                                    each_batch['img128_GT'][0].detach().cpu(), 
                                    img128_fake[0].detach().cpu()))
            
            #print(new_pair.shape)
            #print(img128_fake.shape)
            
            #fake_fname = 'generated-frontal-ep={0:0=3d}_id={0:0=3d}.png'.format(index, each_batch['id'][0].item())
            fake_fname = 'generated-frontal-ep={0:0=3d}.png'.format(index)
            save_image(new_pair, os.path.join(images_save_dir, fake_fname))
            
            #print('Saving', fake_fname)
        
        break

In [64]:
#save_samples(2, G, train_dl, show=True)

In [65]:
G = Generator(num_classes=10)
to_device(G, device)

D = Discriminator()
to_device(D, device)

loss_G = G_Loss()
loss_D = D_Loss()

## Training the Model

In [66]:
def train_discriminator(D, loss_D, opt_d, img128_fake, inputs):
    #Clear discriminator gradients
    opt_d.zero_grad()
    
    #Calculate loss
    loss_d = loss_D(D, img128_fake, inputs)
    
    #Update discriminator weights
    loss_d.backward()
    opt_d.step()
    return loss_d.item()

In [67]:
def train_generator(D, G, loss_G, opt_g, img128_fake, img64_fake, img32_fake, inputs):
    #Clear generator gradients
    opt_g.zero_grad()
    
    #Calculate loss
    loss_g = loss_G(G, D, img128_fake, img64_fake, img32_fake, inputs)
    
    #Update generator weights
    loss_g.backward()
    opt_g.step()
    
    return loss_g.item()

In [68]:
def fit(epochs, G, D, loss_G, loss_D, train_dl, opt_fn=None, lr=None, lr_func=None):
    
    torch.cuda.empty_cache()
    
    train_G_losses, train_D_losses = [], []
    
    #instantiate the optimizer
    if opt_fn is None: opt_fn = torch.optim.Adam
    opt_G = opt_fn(G.parameters(), lr = lr)
    opt_D = opt_fn(D.parameters(), lr = lr)
    
    #scheduler_network = torch.optim.lr_scheduler.LambdaLR(optimizer=opt, lr_lambda=lr_func)
    
    for epoch in range(epochs):
        ep_train_g_losses, ep_train_d_losses, train_len = [], [], []
        
        #Training
        G.train()
        D.train()
        for batch in tqdm.tqdm(train_dl):
            #Generate predictions
            img128_fake, img64_fake, img32_fake = G(batch['img128'], batch['img64'], batch['img32'])
    
            train_d_loss = train_discriminator(D, loss_D, opt_D, img128_fake, batch)
            train_g_loss = train_generator(D, G, loss_G, opt_G, img128_fake, img64_fake, img32_fake, batch)
            len_batch = len(batch)
            
            ep_train_g_losses.append(train_g_loss)
            ep_train_d_losses.append(train_d_loss)
            train_len.append(len_batch) #batch_size
            
        #scheduler_network.step()
        #scheduler_out.step()
        
        total = np.sum(train_len)
        avg_g_train_loss = np.sum(np.multiply(ep_train_g_losses, train_len)) / total
        avg_d_train_loss = np.sum(np.multiply(ep_train_d_losses, train_len)) / total
                
        #Evaluation
        

        #Record the loss
        train_G_losses.append(avg_g_train_loss)
        train_D_losses.append(avg_d_train_loss)

        #Checkpointing the model - saving every 'n' epochs
        checkpoint_path = "../Checkpoints_pair_func/model_" +str(epoch+1)+".pt"
        
        if ((epoch)%5 == 0):
            torch.save({
                'epoch': epoch+1,
                'G_state_dict': G.state_dict(),
                'D_state_dict': D.state_dict(),
                'g_train_loss': avg_g_train_loss,
                'd_train_loss': avg_d_train_loss,
            }, checkpoint_path)
        
        
        #Print progress:
        print('Epoch [{}/{}], Train_G_loss: {:.4f}, Train_D_loss: {:.4f}'
              .format(epoch+1, epochs, avg_g_train_loss, avg_d_train_loss))
        
        save_samples(epoch+1, G, train_dl, show=False)
        
    return train_G_losses, train_D_losses

In [69]:
chckpnt_dir = "Checkpoints_pair_func"
pth_chckpnt_dir = os.path.join(root_dir, chckpnt_dir)

if not os.path.exists(pth_chckpnt_dir):
    os.makedirs(pth_chckpnt_dir)

In [70]:
opt_func = torch.optim.Adam

In [71]:
# #No dynamic updation in LR
# def unit_lr(epoch):
#     return 1

In [72]:
num_epochs = 50
lr = 1e-4

In [ ]:
history = fit(epochs=num_epochs, G=G, D=D, loss_G=loss_G, loss_D=loss_D, 
              train_dl=train_dl, opt_fn=opt_func, lr=lr)

100%|█████████████████████████████████████████| 339/339 [06:01<00:00,  1.07s/it]


Epoch [1/50], Train_G_loss: 0.7808, Train_D_loss: 1.8860


100%|█████████████████████████████████████████| 339/339 [05:59<00:00,  1.06s/it]

Epoch [2/50], Train_G_loss: 0.4485, Train_D_loss: -0.4143



100%|█████████████████████████████████████████| 339/339 [05:59<00:00,  1.06s/it]

...


100%|█████████████████████████████████████████| 339/339 [06:17<00:00,  1.11s/it]

Epoch [39/50], Train_G_loss: 0.2350, Train_D_loss: -0.9094



100%|█████████████████████████████████████████| 339/339 [06:30<00:00,  1.15s/it]

Epoch [40/50], Train_G_loss: 0.2355, Train_D_loss: -0.9018



 36%|██████████████▊                          | 122/339 [02:22<04:20,  1.20s/it]

In [ ]:
losses_g, losses_d = history

In [ ]:
# num_epochs = int(150/5) #150
# init_lr = 1e-4
# lr = [init_lr, init_lr/2, init_lr/4, init_lr/8, init_lr/16]
# losses_g = []
# losses_d = []

# for ad_lr in lr:
#     history = fit(epochs=num_epochs, G=G, D=D, loss_G=loss_G, loss_D=loss_D, 
#               train_dl=train_dl, opt_fn=opt_func, lr=ad_lr)
#     temp_g, temp_d = history
#     losses_g.append(temp_g)
#     losses_d.append(temp_d)

# losses_g = np.concatenate(losses_g)
# losses_d = np.concatenate(losses_d)

## Plot the curves

In [ ]:
def plot_losses(losses_d, losses_g):

    plt.plot(losses_d, '-') #[0:100]
    plt.plot(losses_g, '-') #[0:100]
    
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend(['Discriminator', 'Generator'])
    plt.tick_params(labelcolor='g')
    
    plt.title('Loss vs. No. of epochs')
    plt.show()

In [ ]:
plot_losses(losses_d, losses_g)

In [ ]:
# def plot_losses(epoch, losses_d, losses_g):
#     x = np.arange(1, epoch+1, 1)
    
#     plt.plot(x, losses_d, '-') #[0:100]
#     plt.plot(x, losses_g, '-') #[0:100]
    
#     plt.tick_params(labelcolor='g')
#     plt.xticks(np.linspace(1, epoch, 10))
    
#     plt.xlabel('epoch')
#     plt.ylabel('loss')
#     plt.legend(['Discriminator', 'Generator'])
#     plt.title('Loss vs. No. of epochs')
#     plt.show()

In [ ]:
# plot_losses(150, losses_d, losses_g)

## Load the saved model

In [ ]:
# checkpoint = torch.load("../Checkpoints_pair_func/model_26.pt")
# G.load_state_dict(checkpoint['G_state_dict'])
# D.load_state_dict(checkpoint['D_state_dict'])
# epoch = checkpoint['epoch']
# #opt_G = checkpoint['G_optimizer_state_dict']
# #opt_D = checkpoint['D_optimizer_state_dict']
# #opt_D['state'][0]['momentum_buffer']
# avg_g_train_loss = checkpoint['g_train_loss']
# avg_d_train_loss = checkpoint['d_train_loss']

In [ ]:
# epoch, avg_g_train_loss, avg_d_train_loss

## Displaying results

In [ ]:
def generate_test_outputs(G, test_dl):
        
    with torch.no_grad():
        #pass each batch through the model
        for batch in test_dl:
            #print(each_batch['img128'][0].shape)
            img1 = torch.permute(batch['img128'][0], (1, 2, 0))
            img1 = (img1*127)+127.5
            img1 = img1.type(torch.int64)
    
            #Generate predictions
            img128_fake, img64_fake, img32_fake = G(batch['img128'], batch['img64'], batch['img32'])
            
            img2 = torch.permute(img128_fake[0], (1, 2, 0))
            img2 = (img2*127)+127.5
            img2 = img2.type(torch.int64)
            
            img3 = torch.permute(batch['img128_GT'][0], (1, 2, 0))
            img3 = (img3*127)+127.5
            img3 = img3.type(torch.int64)

            plt.subplot(1, 3, 1)
            plt.imshow(img1)
            plt.axis('off')
            plt.title('Id: ' + str(batch['id'][0].item()))

            plt.subplot(1, 3, 2)
            plt.imshow(img2.detach().cpu().numpy())
            plt.axis('off')
            
            plt.subplot(1, 3, 3)
            plt.imshow(img3)
            plt.axis('off')

            plt.show()

In [ ]:
generate_test_outputs(G, train_dl)

In [ ]:
generate_test_outputs(G, test_dl)

In [ ]:
# # Creating a new data frame
# newDataframe = pd.DataFrame()
# filename = "TP-GAN.xlsx"

# newDataframe['G Train Loss'] = losses_g
# newDataframe['D Train Loss'] = losses_d

# # Converting the data frame to an excel file
# newDataframe.to_excel(filename, index = False)

# # Reading the data from the outputExcelFile
# excelData = pd.read_excel(filename)

# #Printing the data frame
# print(excelData)

## Evaluate the Model

In [ ]:
# def evaluate(G, D, loss_G, loss_D, test_dl):
#     g_losses, d_losses, nums = [], [], []
    
#     with torch.no_grad():
#         #pass each batch through the model
#         for batch in tqdm.tqdm(test_dl):
#             #Generate predictions
#             img128_fake, img64_fake, img32_fake = G(batch['img128'], batch['img64'], batch['img32'])
#             #Calculate loss
#             loss_d = loss_D(D, img128_fake, inputs)
#             #Calculate loss
#             loss_g = loss_G(G, D, img128_fake, img64_fake, img32_fake, inputs)
#             len_batch = len(batch)
            
#             d_losses.append(loss_d.item())
#             g_losses.append(loss_g.item())
#             nums.append(len_batch) #batch_size    

#         #Total size of the dataset
#         total = np.sum(nums)
        
#         #Avg. loss across batches
#         avg_d_loss = np.sum(np.multiply(d_losses, nums))/total
#         avg_g_loss = np.sum(np.multiply(g_losses, nums))/total

#     return avg_d_loss, avg_g_loss, img128_fake, img64_fake, img32_fake

In [ ]:
# #Evaluation
# G.eval()
# D.eval()

# avg_d_loss, avg_g_loss, img128_fake, img64_fake, img32_fake = evaluate(G, D, loss_G, loss_D, train_dl)
# print(avg_d_loss, avg_g_loss)
# print(img128_fake.shape, img64_fake.shape, img32_fake.shape)